In [ ]:
import gc
import torch

def free_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

    print("GPU cache cleared")


In [ ]:
# 약 2분 소요, 이후 세션 재시작 필요
%pip install -q -U uv
!uv pip install --system "vllm==0.17.0" --torch-backend=auto

# 결과


In [ ]:
# 약 4분 소요
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B"

# Load model with vLLM.
llm = LLM(model=model_name, dtype="float16")

# Define the prompt.
prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era."""

# Create sampling parameters.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)

# Time the model generation.
start_time = time.time()
outputs = llm.generate([prompt], sampling_params)
end_time = time.time()

# Print the results.
for output in outputs:
  print(f"Generated text: {output}")
  print(f"Time taken: {end_time - start_time:.2f} seconds")


# results


In [ ]:
if "llm" in globals():
    del llm

free_gpu()


In [ ]:
!nvidia-smi


In [ ]:
import gc
import time
import torch

from vllm import LLM, SamplingParams


model_name = "Qwen/Qwen2.5-0.5B"

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=128,
    seed=42,
)

prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era."""

def benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=1,
    warmup=True,
    repeat=5,
):
    prompts = [prompt] * num_requests

    # 동일 concurrency로 Warm-up
    if warmup:
        llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

        torch.cuda.synchronize()

    times = []
    token_rates = []
    outputs_last = None

    # 여러 번 반복 측정
    for _ in range(repeat):

        torch.cuda.synchronize()

        start = time.perf_counter()

        outputs = llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        output_tokens = sum(
            len(output.outputs[0].token_ids)
            for output in outputs
        )

        times.append(elapsed)
        token_rates.append(
            output_tokens / elapsed
        )

        outputs_last = outputs

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    # Token 수 계산
    input_tokens = sum(
        len(output.prompt_token_ids)
        for output in outputs
    )

    output_tokens = sum(
        len(output.outputs[0].token_ids)
        for output in outputs
    )

    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    return {
        "requests": num_requests,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "time_sec": elapsed,
        "requests_per_sec": num_requests / elapsed,
        "output_tokens_per_sec": output_tokens / elapsed,
        "total_tokens_per_sec":
            (input_tokens + output_tokens) / elapsed,
        "peak_gpu_memory_gb": peak_memory,
    }


def print_result(name, result):
    print(f"\n========== {name} ==========")
    print(f"Requests           : {result['requests']}")
    print(f"Input tokens       : {result['input_tokens']}")
    print(f"Output tokens      : {result['output_tokens']}")
    print(f"Time               : {result['time_sec']:.3f} sec")
    print(f"Requests/sec       : {result['requests_per_sec']:.2f}")
    print(f"Output tokens/sec  : {result['output_tokens_per_sec']:.2f}")
    print(f"Total tokens/sec   : {result['total_tokens_per_sec']:.2f}")
    print(f"Peak GPU memory    : {result['peak_gpu_memory_gb']:.2f} GB")


In [ ]:
load_start = time.perf_counter()

llm = LLM(
    model=model_name,
    dtype="float16",
)

baseline_load_time = time.perf_counter() - load_start

print(f"Baseline load time: {baseline_load_time:.2f} sec")


In [ ]:
baseline_1 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=1,
    repeat=5,
)

print_result(
    "Baseline - Single",
    baseline_1
)


In [ ]:
baseline_32 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=32,
    repeat=5,
)

print_result(
    "Baseline - Batch 32",
    baseline_32
)


In [ ]:
baseline_64 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=64,
    repeat=5,
)

print_result(
    "Baseline - Batch 64",
    baseline_64
)


In [ ]:
baseline_128 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=128,
    repeat=5,
)

print_result(
    "Baseline - Batch 128",
    baseline_128
)


In [ ]:
if "llm" in globals():
    del llm

free_gpu()


In [ ]:
# 튜닝 모델 로드
load_start = time.perf_counter()

llm = LLM(
    model=model_name,
    dtype="float16",

    # Memory / context
    swap_space=4,
    max_model_len=4096,
    gpu_memory_utilization=0.90,

    # PagedAttention / KV Cache
    block_size=16,
    enable_prefix_caching=True,

    # Scheduler
    max_num_seqs=256,

    # Prefill
    enable_chunked_prefill=True,

    # CUDA Graph
    enforce_eager=False,

    # Multi-GPU
    disable_custom_all_reduce=False,
)

tuned_load_time = time.perf_counter() - load_start

print(f"Tuned load time: {tuned_load_time:.2f} sec")


In [ ]:
tuned_1 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=1,
    repeat=5,
)

print_result(
    "Tuned - Single",
    tuned_1
)


In [ ]:
tuned_32 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=32,
    repeat=5,
)

print_result(
    "Tuned - Batch 32",
    tuned_32
)


In [ ]:
tuned_64 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=64,
    repeat=5,
)

print_result(
    "Tuned - Batch 64",
    tuned_64
)


In [ ]:
tuned_128 = benchmark_llm(
    llm,
    prompt,
    sampling_params,
    num_requests=128,
    repeat=5,
)

print_result(
    "Tuned - Batch 128",
    tuned_128
)


In [ ]:
if "llm" in globals():
    del llm

free_gpu()


In [ ]:
def percent_change(old, new):
    return ((new - old) / old) * 100


def latency_improvement(old, new):
    return ((old - new) / old) * 100

# Result

print("\n" + "=" * 65)
print("                 BASELINE vs TUNED")
print("=" * 65)


print("\n[Model Loading]")
print(f"Baseline : {baseline_load_time:.2f} sec")
print(f"Tuned    : {tuned_load_time:.2f} sec")

print(
    f"Change   : "
    f"{latency_improvement(baseline_load_time, tuned_load_time):+.2f}%"
)


print("\n[Single Request]")

print(
    f"Latency       : "
    f"{baseline_1['time_sec']:.3f} -> "
    f"{tuned_1['time_sec']:.3f} sec"
)

print(
    f"Latency Δ     : "
    f"{latency_improvement(
        baseline_1['time_sec'],
        tuned_1['time_sec']
    ):+.2f}%"
)

print(
    f"Output tok/s  : "
    f"{baseline_1['output_tokens_per_sec']:.2f} -> "
    f"{tuned_1['output_tokens_per_sec']:.2f}"
)

print(
    f"Throughput Δ  : "
    f"{percent_change(
        baseline_1['output_tokens_per_sec'],
        tuned_1['output_tokens_per_sec']
    ):+.2f}%"
)


print("\n[Batch 32]")

print(
    f"Time          : "
    f"{baseline_32['time_sec']:.3f} -> "
    f"{tuned_32['time_sec']:.3f} sec"
)

print(
    f"Requests/sec  : "
    f"{baseline_32['requests_per_sec']:.2f} -> "
    f"{tuned_32['requests_per_sec']:.2f}"
)

print(
    f"Output tok/s  : "
    f"{baseline_32['output_tokens_per_sec']:.2f} -> "
    f"{tuned_32['output_tokens_per_sec']:.2f}"
)

print(
    f"Throughput Δ  : "
    f"{percent_change(
        baseline_32['output_tokens_per_sec'],
        tuned_32['output_tokens_per_sec']
    ):+.2f}%"
)

print("\n[Batch 64]")

print(
    f"Time          : "
    f"{baseline_64['time_sec']:.3f} -> "
    f"{tuned_64['time_sec']:.3f} sec"
)

print(
    f"Requests/sec  : "
    f"{baseline_64['requests_per_sec']:.2f} -> "
    f"{tuned_64['requests_per_sec']:.2f}"
)

print(
    f"Output tok/s  : "
    f"{baseline_64['output_tokens_per_sec']:.2f} -> "
    f"{tuned_64['output_tokens_per_sec']:.2f}"
)

print(
    f"Throughput Δ  : "
    f"{percent_change(
        baseline_64['output_tokens_per_sec'],
        tuned_64['output_tokens_per_sec']
    ):+.2f}%"
)

print("\n[Batch 128]")

print(
    f"Time          : "
    f"{baseline_128['time_sec']:.3f} -> "
    f"{tuned_128['time_sec']:.3f} sec"
)

print(
    f"Requests/sec  : "
    f"{baseline_128['requests_per_sec']:.2f} -> "
    f"{tuned_128['requests_per_sec']:.2f}"
)

print(
    f"Output tok/s  : "
    f"{baseline_128['output_tokens_per_sec']:.2f} -> "
    f"{tuned_128['output_tokens_per_sec']:.2f}"
)

print(
    f"Throughput Δ  : "
    f"{percent_change(
        baseline_128['output_tokens_per_sec'],
        tuned_128['output_tokens_per_sec']
    ):+.2f}%"
)
